# **Project 2 - LLMs and Generative AI**

Below is a ready-to-use notebook outline for your assignment. Students will only need to change the prompts in designated cells. The notebook uses Hugging Face Transformers for open-source models, namely MedGemma.

## Install and Import Packages

In [ ]:
!pip install bitsandbytes

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
import torch

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load Generation Model and Tokenizer


**MedGemma Description:**

MedGemma is a collection of Gemma 3 variants that are trained for performance on medical text and image comprehension.

You can find more information here:
https://huggingface.co/unsloth/medgemma-4b-it-bnb-4bit.

In [ ]:
model_name = "unsloth/medgemma-4b-it-bnb-4bit"

print(f"🔄 Loading generation model: {model_name}")

generation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
generation_tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_tokenizer.pad_token = generation_tokenizer.pad_token or generation_tokenizer.eos_token

Function to generate the output using the LLM defined above

In [4]:
def generate(messages):

    text = generation_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = generation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(generation_model.device)

    with torch.inference_mode():
        generation = generation_model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    output = generation_tokenizer.decode(generation[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return output

## Load Notes

**Note 1:** English clinical note

In [5]:
note = """The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis.
She presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.

On exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia.
ECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.
The patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function. C
ardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.

She was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was uncomplicated.
Renal function improved with IV fluids and adjustment of antihypertensive medications.

Discharge medications included: apixaban, lisinopril (dose reduced), and acetaminophen for pain. She was advised to follow up with cardiology and nephrology within one week.
Discharged home with stable vital signs and improved energy levels."""

**Note 2:** Portuguese clinical note

In [6]:
note_pt = """1.a consulta
NHC 118441566
H, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)
Sempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.
Desde que se reformou tem dificuldade em dormir. Dorme 3-4 h
Tem uma empresa. Por vezes ansioso
Faz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.
Antes
Já fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico
DRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016
HTA secundária
Hábitos tabágicos suspensos 2002

Adenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018

Tumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...

Medicação atual: amlodipina e concor . Alprazolam

Refere sensação de calor nos membros inferiores que melhora com movimento, piora no final do dia, por vezes dificultam o adormecer

Roncopatia. Sem apneias identificadas.

Xavier Mourato OM 52847"""

## **Exercise 1: Summarization**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.1 - Intrepertation of the Clinical Note Summary
  <p>Consider the following example for summarize the clinical note.
  
  What can you say about the summary generated? Is the model hallucinating?
  </p>
</div>


In [ ]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.2 - Structured Clinical Notes
  <p>What if we want to structure the clinical Note?
  
  Change the prompt presented in the example before to have the summary of the clinical note in a structured format.
  
  Example of a structured clinical note:
  * Demographics:
  * History:
  * Diagnosis:
  * ...
  </p>
</div>

In [ ]:
# messages = [
#     {
#         "role": "system",
#         ...
# ]

output = generate(messages)
print(output)

## **Exercise 2: Question-Answering**
We can use the LLM to answer specific questions regarding the clinical note.

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.1 - Patient status on Exam
  <p>How was the patient feeling in the moment of the exame?

  Change the prompt and interpret the output.
  </p>
</div>

In [ ]:
# messages = [
#     {
#         "role": "system",
#         ...
# ]

output = generate(messages)
print(output)

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.2 - Discharge Medication
  <p>Which drugs are included in the discharge medications?

  Change the prompt and interpret the output. Try include the doses if they are available.
  </p>
</div>

In [ ]:
# messages = [
#     {
#         "role": "system",
#         ...
# ]

output = generate(messages)
print(output)

## **Exercise 3 - Entity Extraction**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.1 - Procedures Extraction
  <p>Still using the same LLM, find the procedures mentioned in the clinical note?

  Change the prompt and interpret the output.
  </p>
</div>

In [ ]:
# messages = [
#     {
#         "role": "system",
#         ...
# ]

output = generate(messages)
print(output)

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.2 - Conditions Extraction for clinical note in portuguese
  <p>Still using the same LLM, find the conditions mentioned in the clinical note written in portuguese?

  Change the prompt and interpret the output. The portuguese note is saved in the variable 'note_pt'.
  </p>
</div>

In [ ]:
# messages = [
#     {
#         "role": "system",
#         ...
# ]

output = generate(messages)
print(output)

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.3 - Named-Entity Recognition (NER) Model
  <p> These type of models are trained to identify entities in the clinical notes. Below we show how to use a NER model to extract entities, in this case conditions, from the clinical note.

  Considering the conditions identified, which are the main differences comparing to the output generated in the previous exercise?
  </p>
</div>

In [24]:
entity_extractor = pipeline('ner', model='portugueseNLP/medialbertina_pt-pt_900m_NER', aggregation_strategy="simple")
entities = entity_extractor(note_pt)

Device set to use cuda:0


In [ ]:
diagnosis = [annotation['word'] for annotation in entities if annotation['entity_group'] == 'Diagnostico']
diagnosis

**Model Description:**

MediAlbertina PT-PT 900M NER was created through fine-tuning of MediAlbertina PT-PT 900M on real European Portuguese EMRs that have been hand-annotated for the following entities:

- Diagnostico (D): All types of diseases and conditions following the ICD-10-CM guidelines.
- Sintoma (S): Any complaints or evidence from healthcare professionals indicating that a patient is experiencing a medical condition.
Medicamento (M): Something that is administrated to the patient (through any route), including drugs, specific food/drink, vitamins, or blood for transfusion.
- Dosagem (D): Dosage and frequency of medication administration.
ProcedimentoMedico (PM): Anything healthcare professionals do related to patients, including exams, moving patients, administering something, or even surgeries.
- SinalVital (SV): Quantifiable indicators in a patient that can be measured, always associated with a specific result. Examples include cholesterol levels, diuresis, weight, or glycaemia.
- Resultado (R): Results can be associated with Medical Procedures and Vital Signs. It can be a numerical value if something was measured (e.g., the value associated with blood pressure) or a descriptor to indicate the result (e.g., positive/negative, functional).
- Progresso (P): Describes the progress of patient’s condition. Typically, it includes verbs like improving, evolving, or regressing and mentions to patient’s stability.

https://huggingface.co/portugueseNLP/medialbertina_pt-pt_900m_NER